# Endpoint analysis — QuPath measurements

**Template** — edit the *Configuration* cell and run all cells.

Selects the last timepoint from a QuPath TSV and produces per-condition boxplots.
For paired experimental designs (same animal/sample in multiple conditions), use `paired_boxplot`;
for unpaired designs, use `boxplot`.

In [ ]:
import pathlib, sys
ROOT = pathlib.Path().resolve().parents[1]  # repo root; adjust if notebook is not in notebooks/<subdir>/
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import hcs_analysis as hcs

## Configuration

In [ ]:
# ── EDIT THESE ──────────────────────────────────────────────────────────────
RESULTS_TSV   = ROOT / "data" / "TODO_experiment" / "Results" / "measurements.tsv"
PLATE_MAP_CSV = ROOT / "data" / "TODO_experiment" / "plate_map.csv"
PLATE_IDS     = None          # multi-plate: ["VID9955", "VID9960"]; None for single-plate

INTERVAL_MIN  = 30            # minutes between QuPath time indices
START_MIN     = 0             # elapsed_min for time index 0

CLASSIFICATION = None         # keep only one class, e.g. "Neurosphere"; None keeps all
METRICS = ["Circularity", "Area px^2"]  # measurement columns to plot

PAIRED = True                 # True: paired_boxplot connecting same sample across conditions
                              # False: unpaired boxplot
# ────────────────────────────────────────────────────────────────────────────

## 1. Load data and select endpoint

In [ ]:
df = hcs.load_qupath(RESULTS_TSV, interval_min=INTERVAL_MIN, start_min=START_MIN)
plate_map = hcs.load_plate_map(PLATE_MAP_CSV, plate_ids=PLATE_IDS)
df = hcs.merge_plate_map(df, plate_map)

if CLASSIFICATION is not None:
    df = hcs.filter_by(df, Classification=CLASSIFICATION)

# Keep only the last timepoint
endpoint = df["elapsed_min"].max()
df = hcs.filter_timepoints(df, [endpoint])

print(f"{len(df):,} objects  |  {df['Well'].nunique()} wells  |  endpoint: {endpoint} min ({endpoint/60:.1f} h)")
df.head(3)

## 2. QC — object counts

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
hcs.qc_counts(df, groupby="Well", ax=ax)
ax.set_title(f"Object count per well at endpoint ({endpoint} min)")
plt.tight_layout()

## 3. Aggregate

For a paired design: average technical replicates → one value per (sample, condition) before plotting.

In [ ]:
df_well = hcs.to_well_means(df, metrics=METRICS)

# Average technical replicates within each (sample, condition).
# For an unpaired design with no technical replicates, df_well is already at the right level.
df_bio = df_well.groupby(["sample", "condition"])[METRICS].mean().reset_index()

print(f"n per condition:")
print(df_bio.groupby("condition")["sample"].nunique())
df_bio

## 4. Endpoint plots

In [ ]:
fig, axes = plt.subplots(1, len(METRICS), figsize=(4 * len(METRICS), 5))
if len(METRICS) == 1:
    axes = [axes]
for ax, metric in zip(axes, METRICS):
    if PAIRED:
        hcs.paired_boxplot(df_bio, y=metric, condition_col="condition", pair_col="sample", ax=ax)
    else:
        hcs.boxplot(df_bio, y=metric, x="condition", ax=ax)
    ax.set_title(metric)
plt.tight_layout()